# 0. Optional Steps

The steps in this section are optional and used for debug logging

## 0.1 Set the environment to use a proxy

In [ ]:
import os

# Setup proxy through local proxy; bypass proxy for metadata endpoint to avoid any issues with Microsoft authentication libraries
os.environ["HTTP_PROXY"] = "http://127.0.0.1:8000"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:8000"
os.environ["NO_PROXY"] = "169.254.169.254"

# Add private CA root cert to trusted cert bundle for SDKs that use requests and httpx
os.environ["REQUESTS_CA_BUNDLE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"
os.environ["SSL_CERT_FILE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"

import os

# Setup proxy through local proxy; bypass proxy for metadata endpoint to avoid any issues with Microsoft authentication libraries
os.environ["HTTP_PROXY"] = "http://127.0.0.1:8000"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:8000"
os.environ["NO_PROXY"] = "169.254.169.254"

# Add private CA root cert to trusted cert bundle for SDKs that use requests and httpx
os.environ["REQUESTS_CA_BUNDLE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"
os.environ["SSL_CERT_FILE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"

# 1. Setup the notebook

The steps in this section setup the notebook

## 1.1 Install required libraries

In [ ]:
%pip install -r requirements.txt

## 1.2 Setup a logger

In [ ]:
# Setup a Python logger
import logging
import sys

def configure_logging(level="ERROR"):
    """This function configures logging for code being run based on the specified level.
    Args:
        level (str): The logging level to use (e.g., "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL").
    """

    try:
        # Convert the level string to uppercase so it matches what the logging library expects
        logging_level = getattr(logging, level.upper(), None)

        # Setup a logging format
        logging.basicConfig(
            level=logging_level,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[logging.StreamHandler(sys.stdout)]
        )
    except Exception as e:
        print(f"Failed to set up logging: {e}", file=sys.stderr)
        sys.exit(1) 

configure_logging("ERROR")

## 1.3 Load environment variables

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(".env")

FOUNDRY_MODEL_ENDPOINT = os.getenv("FOUNDRY_MODEL_ENDPOINT")
ENTRA_TENANT_ID = os.getenv("ENTRA_TENANT_ID")
BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID = os.getenv("BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID")
BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET = os.getenv("BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET")
BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID = os.getenv("BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID")
BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET = os.getenv("BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET")
BATCH_DEPLOYMENT_NAME = os.getenv("BATCH_DEPLOYMENT_NAME")


## 1.4 Update the sample file with the deployment name



In [ ]:
# Read the sample file as plain text
with open("sample.jsonl", "r") as f:
    file_text = f.read()

# Replace the DEPLOYMENT_NAME placeholder with the real deployment name
file_text = file_text.replace("DEPLOYMENT_NAME", BATCH_DEPLOYMENT_NAME)

# Save the updated text back to the sample file
with open("sample.jsonl", "w") as f:
    f.write(file_text)

print("sample.jsonl now uses deployment:", BATCH_DEPLOYMENT_NAME)

# 2. Perform batch chat completions

In [ ]:
# Get an Entra token

from azure.identity import ClientSecretCredential, get_bearer_token_provider

credential = ClientSecretCredential(
    client_id=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID,
    client_secret=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET,
    tenant_id=ENTRA_TENANT_ID
)

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default"
)
print(token_provider())


In [ ]:
# Upload a file to Foundry

from openai import OpenAI

client = OpenAI(
    base_url=FOUNDRY_MODEL_ENDPOINT,
    api_key=token_provider
)

file = client.files.create(
    file=open("sample.jsonl", 'rb'),
    purpose="batch"
)

print(file.model_dump_json(indent=2))
file_id = file.id

In [ ]:
# Loop until the file is processsed

import datetime
import time

status = client.files.retrieve(file_id).status
print(f"{datetime.datetime.now()} - File Id: {file_id}, Status: {status}")
while status not in ("processed"):
    time.sleep(30)
    file = client.files.retrieve(file_id)
    status = file.status
    print(f"{datetime.datetime.now()} - File Id: {file_id}, Status: {status}")


In [ ]:
# Create the batch

batch_response = client.batches.create(
        input_file_id=file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h"
)

batch_id = batch_response.id
print(batch_response.model_dump_json(indent=2))

In [ ]:
# Loop until the batch is completed, failed, or cancelled

status = batch_response.status
while status not in ("completed", "failed", "cancelled"):
    print(f"{datetime.datetime.now()
                } - Batch Id: {batch_id}, Status: {status}")
    time.sleep(30)
    batch_response = client.batches.retrieve(batch_id)
    status = batch_response.status
    print(f"{datetime.datetime.now()
                } - Batch Id: {batch_id}, Status: {status}")
    
    if batch_response.status == "failed":
        for error in batch_response.errors.data:
            print(f"Error code: {error.code} Message {error.message}")

output_file_id = batch_response.output_file_id

In [ ]:
# Retrieve and display the output or error file of the batch

if not output_file_id:
    output_file_id = batch_response.error_file_id
if output_file_id:
    file_response = client.files.content(output_file_id)
    raw_responses = file_response.text.strip().split('\n')  

    for raw_response in raw_responses:  
        json_response = json.loads(raw_response)  
        formatted_json = json.dumps(json_response, indent=2)  
        print(formatted_json)